In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import csv
import shutil
import tarfile

from skimage import data, io
from skimage.color import rgb2gray
from skimage.util import img_as_float, img_as_ubyte

# Project folders
project_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

background_dir = project_dir / "data" / "background_research_examples"
calibration_dir = project_dir / "data" / "calibration"
test_dir = project_dir / "data" / "test"

for folder in [background_dir, calibration_dir, test_dir]:
    folder.mkdir(parents=True, exist_ok=True)

print(background_dir)
print(calibration_dir)
print(test_dir)


D:\MScProject\data\background_research_examples
D:\MScProject\data\calibration
D:\MScProject\data\test


In [2]:
# Background research examples
background_images = {
    "camera": data.camera(),
    "astronaut": data.astronaut(),
    "coins": data.coins(),
    "moon": data.moon(),
    "rocket": data.rocket(),
    "grass": data.grass(),
    "gravel": data.gravel(),
    "clock": data.clock(),
    "text": data.text(),
    "brick": data.brick(),
    "cat": data.cat(),
    "checkerboard": data.checkerboard(),
    "coffee": data.coffee(),
    "page": data.page(),
    "phantom": data.shepp_logan_phantom(),
    "eagle": data.eagle(),
    "motorcycle": data.stereo_motorcycle()[0]
}

for image_name, image in background_images.items():
    image = img_as_float(image)

    if image.ndim == 3:
        image = rgb2gray(image)

    save_path = background_dir / f"{image_name}.png"
    io.imsave(save_path, img_as_ubyte(image))

print(f"Saved {len(background_images)} background research images.")


Saved 17 background research images.


In [3]:
# Calibration images
dataset_url = (
    "https://www2.eecs.berkeley.edu/Research/Projects/"
    "CS/vision/bsds/BSDS300-images.tgz"
)

archive_path = project_dir / "data" / "BSDS300-images.tgz"
mapping_path = calibration_dir / "filename_mapping.csv"

if not archive_path.exists():
    urlretrieve(dataset_url, archive_path)

mapping_rows = []

with tarfile.open(archive_path, "r:gz") as archive:
    training_members = sorted(
        [
            member for member in archive.getmembers()
            if "/images/train/" in member.name
            and member.name.lower().endswith(".jpg")
        ],
        key=lambda member: Path(member.name).stem
    )

    for index, member in enumerate(training_members, start=1):
        source = archive.extractfile(member)

        if source is not None:
            new_name = f"{index:03d}.jpg"
            original_name = Path(member.name).name
            output_path = calibration_dir / new_name

            if not output_path.exists():
                with output_path.open("wb") as destination:
                    shutil.copyfileobj(source, destination)

            mapping_rows.append([new_name, original_name])

with mapping_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["calibration_name", "original_name"])
    writer.writerows(mapping_rows)

print(f"Saved {len(training_members)} calibration images.")


Download completed.
Saved 200 training images.
Folder: D:\MScProject\notebooks\data\calibration
First image: 001.jpg
Last image: 200.jpg
Filename mapping: data\calibration\filename_mapping.csv


In [4]:
# Test images
mapping_path = test_dir / "filename_mapping.csv"
mapping_rows = []

with tarfile.open(archive_path, "r:gz") as archive:
    test_members = sorted(
        [
            member for member in archive.getmembers()
            if "/images/test/" in member.name
            and member.name.lower().endswith(".jpg")
        ],
        key=lambda member: Path(member.name).stem
    )

    if len(test_members) != 100:
        raise ValueError(f"Expected 100 test images, but found {len(test_members)}.")

    for index, member in enumerate(test_members, start=1):
        source = archive.extractfile(member)

        if source is not None:
            new_name = f"{index:03d}.jpg"
            original_name = Path(member.name).name
            output_path = test_dir / new_name

            if not output_path.exists():
                with output_path.open("wb") as destination:
                    shutil.copyfileobj(source, destination)

            mapping_rows.append([new_name, original_name])

with mapping_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["test_name", "original_name"])
    writer.writerows(mapping_rows)

print(f"Saved {len(test_members)} test images.")


Saved 100 test images.
Folder: D:\MScProject\notebooks\data\test
First image: 001.jpg
Last image: 100.jpg
Filename mapping: data\test\filename_mapping.csv
